# 从零实现 Speculative Decoding：提议、验证、拒绝残差与 cache rollback

本 Notebook 用手写 BigramLM57 与 SpeculativeDecoder57 展示 speculative sampling 的完整概率语义：draft 顺序提议，target 以 min(1,p/q) 验证，拒绝时从归一化正部 (p-q)+ 采样替代 token，全接受时再由 target 产生 bonus token。

实现显式处理 EOS、torch.Generator、请求级 cache 位置与拒绝回滚。它不用 transformers、vLLM 或现成 speculative decoder；小词表 bigram 只是精确概率 oracle，不代表真实 LLM 的吞吐收益。

In [ ]:
import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。

SEED57 = 5701  # 计算并保存当前步骤的中间状态。
random.seed(SEED57); np.random.seed(SEED57); torch.manual_seed(SEED57)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE57 = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

def canonical_json57(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 返回当前分支计算出的结果。

def sha57(raw):  # 定义本节可复用的核心函数。
    return hashlib.sha256(raw).hexdigest()  # 返回当前分支计算出的结果。

assert DEVICE57.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。

## 1. Token、评估 prompt 与概率矩阵

词表只有 BOS、A、B、EOS。target/draft 都是一阶因果模型，transition[current,next] 每行和为 1。draft 故意在 BOS 后过度偏好 B，让拒绝路径可观测；target/draft tokenizer 必须完全相同。

PROMPTS57 与 SPLIT57 会被完整写入发布 metadata。calibration 可用于选择 gamma，test 只做机制回归；这里没有声称对真实请求分布有效。

In [ ]:
TOKENIZER57 = {"<bos>": 0, "A": 1, "B": 2, "<eos>": 3}  # 计算并保存当前步骤的中间状态。
PROMPTS57 = [  # 计算并保存当前步骤的中间状态。
    {"id": "cal-bos", "tokens": [0]},  # 执行当前语句以推进本节示例。
    {"id": "cal-a", "tokens": [0, 1]},  # 执行当前语句以推进本节示例。
    {"id": "test-b", "tokens": [0, 2]},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
SPLIT57 = {"calibration": ["cal-bos", "cal-a"], "test": ["test-b"]}  # 计算并保存当前步骤的中间状态。
TARGET_PROBS57 = torch.tensor([  # 计算并保存当前步骤的中间状态。
    [0.02, 0.52, 0.38, 0.08],  # 执行当前语句以推进本节示例。
    [0.02, 0.10, 0.30, 0.58],  # 执行当前语句以推进本节示例。
    [0.02, 0.25, 0.15, 0.58],  # 执行当前语句以推进本节示例。
    [0.01, 0.01, 0.01, 0.97],  # 执行当前语句以推进本节示例。
], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
DRAFT_PROBS57 = torch.tensor([  # 计算并保存当前步骤的中间状态。
    [0.02, 0.25, 0.65, 0.08],  # 执行当前语句以推进本节示例。
    [0.02, 0.15, 0.20, 0.63],  # 执行当前语句以推进本节示例。
    [0.02, 0.45, 0.08, 0.45],  # 执行当前语句以推进本节示例。
    [0.01, 0.01, 0.01, 0.97],  # 执行当前语句以推进本节示例。
], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(TARGET_PROBS57.sum(-1), torch.ones(4, dtype=torch.float64))  # 用受控断言验证关键不变量。
assert torch.allclose(DRAFT_PROBS57.sum(-1), torch.ones(4, dtype=torch.float64))  # 用受控断言验证关键不变量。
assert (TARGET_PROBS57 >= 0).all() and (DRAFT_PROBS57 >= 0).all()  # 用受控断言验证关键不变量。
assert set(sum(SPLIT57.values(), [])) == {row["id"] for row in PROMPTS57}  # 用受控断言验证关键不变量。
assert not (set(SPLIT57["calibration"]) & set(SPLIT57["test"]))  # 用受控断言验证关键不变量。

## 2. 手写 causal bigram nn.Module

TransitionHead57 保存 logits:[V,V]；给定 last_tokens:[B] 返回 next_logits:[B,V]。BigramLM57 对 input_ids:[B,T] 的每个位置独立查表，因此 logits[:,t] 只条件于当前位置 token，不偷看未来。

真实 LLM 会用多层 attention 和 KV cache；选择 bigram 是为了能精确穷举 p/q，而不是逃避自回归接口、请求位置与缓存语义。

In [ ]:
class TransitionHead57(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, probabilities):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if probabilities.ndim != 2 or probabilities.shape[0] != probabilities.shape[1]:  # 按当前条件选择后续控制路径。
            raise ValueError("transition_must_be_square")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(probabilities).all() or (probabilities < 0).any():  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_transition_probabilities")  # 遇到非法合同立即显式失败。
        if not torch.allclose(probabilities.sum(-1), torch.ones(probabilities.shape[0], dtype=probabilities.dtype), atol=1e-8):  # 按当前条件选择后续控制路径。
            raise ValueError("transition_rows_must_sum_to_one")  # 遇到非法合同立即显式失败。
        logits = probabilities.clamp_min(1e-30).log().float()  # 计算并保存当前步骤的中间状态。
        self.logits = nn.Parameter(logits, requires_grad=False)  # 计算并保存当前步骤的中间状态。
        self.vocab_size = probabilities.shape[0]  # 计算并保存当前步骤的中间状态。

    def forward(self, last_tokens):  # 定义本节可复用的核心函数。
        if last_tokens.dtype != torch.long or last_tokens.ndim < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("last_tokens_must_be_long")  # 遇到非法合同立即显式失败。
        if last_tokens.min() < 0 or last_tokens.max() >= self.vocab_size:  # 按当前条件选择后续控制路径。
            raise ValueError("token_id_out_of_range")  # 遇到非法合同立即显式失败。
        return self.logits[last_tokens]  # 返回当前分支计算出的结果。

class BigramLM57(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, probabilities):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.transition = TransitionHead57(probabilities)  # 计算并保存当前步骤的中间状态。
        self.vocab_size = self.transition.vocab_size  # 计算并保存当前步骤的中间状态。

    def forward(self, input_ids):  # 定义本节可复用的核心函数。
        if input_ids.ndim != 2 or input_ids.shape[1] < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("input_ids_must_have_shape_B_T")  # 遇到非法合同立即显式失败。
        return self.transition(input_ids)  # 返回当前分支计算出的结果。

    @torch.no_grad()  # 为下方定义附加声明式配置。
    def next_probs(self, token):  # 定义本节可复用的核心函数。
        token_tensor = torch.tensor([token], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
        return self.transition(token_tensor)[0].softmax(-1).double()  # 返回当前分支计算出的结果。

target_model57 = BigramLM57(TARGET_PROBS57)  # 计算并保存当前步骤的中间状态。
draft_model57 = BigramLM57(DRAFT_PROBS57)  # 计算并保存当前步骤的中间状态。
model_probe57 = target_model57(torch.tensor([[0, 1, 2]]))  # 计算并保存当前步骤的中间状态。
assert model_probe57.shape == (1, 3, 4)  # 用受控断言验证关键不变量。
assert torch.allclose(target_model57.next_probs(0), TARGET_PROBS57[0], atol=1e-7)  # 用受控断言验证关键不变量。
assert not any(p.requires_grad for p in target_model57.parameters())  # 用受控断言验证关键不变量。

## 3. 为什么接受概率与拒绝残差能恢复 target

draft 先以 q(x) 提议 x，接受概率 a(x)=min(1,p(x)/q(x))。被接受的质量为 q(x)a(x)=min(p(x),q(x))。总拒绝质量 Z=sum(q-p)+，而归一化残差 r(x)=(p(x)-q(x))+/Z。

因为 p、q 都归一化，sum(q-p)+=sum(p-q)+=Z，所以最终首个输出质量为 min(p,q)+Zr=p。下面对词表中每个上下文精确计算，不依赖蒙特卡洛碰巧接近。

In [ ]:
def validate_distribution57(probabilities, name):  # 定义本节可复用的核心函数。
    if probabilities.ndim != 1 or not torch.isfinite(probabilities).all() or (probabilities < 0).any():  # 按当前条件选择后续控制路径。
        raise ValueError(name + "_invalid")  # 遇到非法合同立即显式失败。
    if not torch.allclose(probabilities.sum(), torch.tensor(1.0, dtype=probabilities.dtype), atol=1e-8):  # 按当前条件选择后续控制路径。
        raise ValueError(name + "_not_normalized")  # 遇到非法合同立即显式失败。

def residual_distribution57(p, q):  # 定义本节可复用的核心函数。
    validate_distribution57(p, "target"); validate_distribution57(q, "draft")  # 执行当前语句以推进本节示例。
    positive = (p - q).clamp_min(0)  # 计算并保存当前步骤的中间状态。
    mass = positive.sum()  # 计算并保存当前步骤的中间状态。
    if mass <= 1e-15:  # 按当前条件选择后续控制路径。
        raise RuntimeError("residual_requested_when_p_equals_q")  # 遇到非法合同立即显式失败。
    return positive / mass  # 返回当前分支计算出的结果。

def exact_first_output57(p, q):  # 定义本节可复用的核心函数。
    validate_distribution57(p, "target"); validate_distribution57(q, "draft")  # 执行当前语句以推进本节示例。
    accepted_mass = torch.minimum(p, q)  # 计算并保存当前步骤的中间状态。
    reject_mass = (q - p).clamp_min(0).sum()  # 计算并保存当前步骤的中间状态。
    if reject_mass <= 1e-15:  # 按当前条件选择后续控制路径。
        return accepted_mass  # 返回当前分支计算出的结果。
    return accepted_mass + reject_mass * residual_distribution57(p, q)  # 返回当前分支计算出的结果。

for context57 in range(len(TOKENIZER57)):  # 遍历输入元素以累积或检查结果。
    exact57 = exact_first_output57(TARGET_PROBS57[context57], DRAFT_PROBS57[context57])  # 计算并保存当前步骤的中间状态。
    assert torch.allclose(exact57, TARGET_PROBS57[context57], atol=1e-12)  # 用受控断言验证关键不变量。
    assert torch.allclose(exact57.sum(), torch.tensor(1.0, dtype=torch.float64))  # 用受控断言验证关键不变量。

p_zero_case57 = torch.tensor([0.70, 0.30, 0.0], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
q_zero_case57 = torch.tensor([0.0, 1.0, 0.0], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(exact_first_output57(p_zero_case57, q_zero_case57), p_zero_case57, atol=1e-12)  # 用受控断言验证关键不变量。
assert residual_distribution57(p_zero_case57, q_zero_case57).tolist() == [1.0, 0.0, 0.0]  # 用受控断言验证关键不变量。

## 4. 单 token 验证、q=0 与全接受/全拒绝

实际 proposal 必须来自 q；若某 token 的 q=0 却声称 draft 提议了它，说明采样器或数值管线违约，应 fail closed，而不是静默除零。全接受时原样输出 proposals；第一次拒绝后只输出已接受前缀和一个 residual replacement，并结束当前 block。

In [ ]:
def sample_categorical57(probabilities, generator):  # 定义本节可复用的核心函数。
    validate_distribution57(probabilities, "sampling")  # 执行当前语句以推进本节示例。
    if not isinstance(generator, torch.Generator):  # 按当前条件选择后续控制路径。
        raise TypeError("explicit_torch_generator_required")  # 遇到非法合同立即显式失败。
    return int(torch.multinomial(probabilities.float(), 1, generator=generator).item())  # 返回当前分支计算出的结果。

def acceptance_probability57(p, q, proposal):  # 定义本节可复用的核心函数。
    validate_distribution57(p, "target"); validate_distribution57(q, "draft")  # 执行当前语句以推进本节示例。
    if not isinstance(proposal, int) or not 0 <= proposal < len(p):  # 按当前条件选择后续控制路径。
        raise ValueError("proposal_out_of_range")  # 遇到非法合同立即显式失败。
    if q[proposal] <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("draft_proposed_zero_probability_token")  # 遇到非法合同立即显式失败。
    return min(1.0, float(p[proposal] / q[proposal]))  # 返回当前分支计算出的结果。

def verify_block57(proposals, p_rows, q_rows, uniforms, generator):  # 定义本节可复用的核心函数。
    if not (len(proposals) == len(p_rows) == len(q_rows) == len(uniforms)):  # 按当前条件选择后续控制路径。
        raise ValueError("verify_block_lengths_mismatch")  # 遇到非法合同立即显式失败。
    emitted, accepted = [], 0  # 计算并保存当前步骤的中间状态。
    for proposal, p, q, uniform in zip(proposals, p_rows, q_rows, uniforms):  # 遍历输入元素以累积或检查结果。
        if not 0 <= uniform < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("uniform_out_of_range")  # 遇到非法合同立即显式失败。
        if uniform < acceptance_probability57(p, q, proposal):  # 按当前条件选择后续控制路径。
            emitted.append(proposal); accepted += 1  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            replacement = sample_categorical57(residual_distribution57(p, q), generator)  # 计算并保存当前步骤的中间状态。
            emitted.append(replacement)  # 执行当前语句以推进本节示例。
            return emitted, accepted, True  # 返回当前分支计算出的结果。
    return emitted, accepted, False  # 返回当前分支计算出的结果。

equal_dist57 = torch.tensor([0.2, 0.8], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
all_accept57 = verify_block57([1, 0], [equal_dist57, equal_dist57], [equal_dist57, equal_dist57], [0.99, 0.2], torch.Generator().manual_seed(1))  # 计算并保存当前步骤的中间状态。
assert all_accept57 == ([1, 0], 2, False)  # 用受控断言验证关键不变量。
reject_p57 = torch.tensor([1.0, 0.0], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
reject_q57 = torch.tensor([0.0, 1.0], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
all_reject57 = verify_block57([1], [reject_p57], [reject_q57], [0.5], torch.Generator().manual_seed(2))  # 计算并保存当前步骤的中间状态。
assert all_reject57 == ([0], 0, True)  # 用受控断言验证关键不变量。
assert acceptance_probability57(reject_p57, reject_q57, 1) == 0.0  # 用受控断言验证关键不变量。
boundary_p57 = torch.tensor([1.0, 0.0], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
boundary_q57 = torch.tensor([0.5, 0.5], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
boundary_reject57 = verify_block57([1], [boundary_p57], [boundary_q57], [0.0], torch.Generator().manual_seed(3))  # 计算并保存当前步骤的中间状态。
assert boundary_reject57 == ([0], 0, True)  # 用受控断言验证关键不变量。
ratio_p57 = torch.tensor([0.60, 0.40], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
ratio_q57 = torch.tensor([0.30, 0.70], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
assert acceptance_probability57(ratio_p57, ratio_q57, 0) == 1.0  # 用受控断言验证关键不变量。
assert abs(acceptance_probability57(ratio_p57, ratio_q57, 1) - 4 / 7) < 1e-12  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    acceptance_probability57(reject_p57, reject_q57, 0)  # 执行当前语句以推进本节示例。
    raise AssertionError("q=0 proposal was accepted")  # 遇到非法合同立即显式失败。
except ValueError as error57:  # 捕获预期异常并验证失败分支。
    assert str(error57) == "draft_proposed_zero_probability_token"  # 用受控断言验证关键不变量。

## 5. 请求级 cache 与 rollback

TokenCache57 用 request_id 隔离状态。position 定义为当前已提交 token 数；draft 可在一个 block 内超前，target 拒绝后必须把 draft 回滚到“已接受前缀”，再把 residual replacement 同时追加到两边。

真实 KV cache 回滚的是每层 K/V 页和有效长度，而不是 token list；但位置不变量相同：一个请求的 rollback 不能影响另一个请求，也不能让 target/draft 位置分叉。

In [ ]:
class TokenCache57:  # 定义承载本节状态与行为的数据结构。
    def __init__(self):  # 定义本节可复用的核心函数。
        self._requests = {}  # 计算并保存当前步骤的中间状态。

    def begin(self, request_id, prefix):  # 定义本节可复用的核心函数。
        if not isinstance(request_id, str) or not request_id:  # 按当前条件选择后续控制路径。
            raise ValueError("request_id_must_be_nonempty_string")  # 遇到非法合同立即显式失败。
        if request_id in self._requests:  # 按当前条件选择后续控制路径。
            raise RuntimeError("request_already_exists")  # 遇到非法合同立即显式失败。
        if not prefix:  # 按当前条件选择后续控制路径。
            raise ValueError("prefix_must_be_nonempty")  # 遇到非法合同立即显式失败。
        self._requests[request_id] = list(prefix)  # 计算并保存当前步骤的中间状态。

    def append(self, request_id, token):  # 定义本节可复用的核心函数。
        if request_id not in self._requests:  # 按当前条件选择后续控制路径。
            raise KeyError("unknown_request")  # 遇到非法合同立即显式失败。
        self._requests[request_id].append(int(token))  # 执行当前语句以推进本节示例。

    def rollback(self, request_id, position):  # 定义本节可复用的核心函数。
        if request_id not in self._requests:  # 按当前条件选择后续控制路径。
            raise KeyError("unknown_request")  # 遇到非法合同立即显式失败。
        if not 1 <= position <= len(self._requests[request_id]):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_rollback_position")  # 遇到非法合同立即显式失败。
        removed = len(self._requests[request_id]) - position  # 计算并保存当前步骤的中间状态。
        del self._requests[request_id][position:]  # 执行当前语句以推进本节示例。
        return removed  # 返回当前分支计算出的结果。

    def tokens(self, request_id):  # 定义本节可复用的核心函数。
        if request_id not in self._requests:  # 按当前条件选择后续控制路径。
            raise KeyError("unknown_request")  # 遇到非法合同立即显式失败。
        return list(self._requests[request_id])  # 返回当前分支计算出的结果。

    def position(self, request_id):  # 定义本节可复用的核心函数。
        return len(self.tokens(request_id))  # 返回当前分支计算出的结果。

    def release(self, request_id):  # 定义本节可复用的核心函数。
        if request_id not in self._requests:  # 按当前条件选择后续控制路径。
            raise KeyError("unknown_request")  # 遇到非法合同立即显式失败。
        del self._requests[request_id]  # 执行当前语句以推进本节示例。

cache_probe57 = TokenCache57()  # 计算并保存当前步骤的中间状态。
cache_probe57.begin("r-a", [0]); cache_probe57.begin("r-b", [0, 1])  # 执行当前语句以推进本节示例。
cache_probe57.append("r-a", 2); cache_probe57.append("r-a", 2)  # 执行当前语句以推进本节示例。
removed57 = cache_probe57.rollback("r-a", 2)  # 计算并保存当前步骤的中间状态。
assert removed57 == 1 and cache_probe57.tokens("r-a") == [0, 2]  # 用受控断言验证关键不变量。
assert cache_probe57.tokens("r-b") == [0, 1]  # 用受控断言验证关键不变量。
assert cache_probe57.position("r-a") == cache_probe57.position("r-b") == 2  # 用受控断言验证关键不变量。

## 6. Speculative propose → verify → residual/bonus

gamma 是每个 block 的最大 proposal 数。draft 串行产生候选；target 在教学代码里逐个算 p 以便阅读。若 proposal 被接受，就提交到 target cache；拒绝则从 residual 采样 replacement、回滚 draft 并结束 block。全部接受且预算/EOS 允许时，target 再产生一个 bonus token。

输出长度严格不超过 max_new_tokens。EOS 一经提交立即停止，EOS proposal 被拒绝时则按 replacement 继续，这两个分支不能混淆。

In [ ]:
class SpeculativeDecoder57(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, target, draft, gamma=3, eos_token=3):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if target.vocab_size != draft.vocab_size or gamma < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("incompatible_models_or_gamma")  # 遇到非法合同立即显式失败。
        if not 0 <= eos_token < target.vocab_size:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_eos_token")  # 遇到非法合同立即显式失败。
        self.target, self.draft = target, draft  # 计算并保存当前步骤的中间状态。
        self.gamma, self.eos_token = int(gamma), int(eos_token)  # 计算并保存当前步骤的中间状态。
        self.target_cache, self.draft_cache = TokenCache57(), TokenCache57()  # 计算并保存当前步骤的中间状态。

    @torch.no_grad()  # 为下方定义附加声明式配置。
    def forward(self, prefix, max_new_tokens, generator, request_id):  # 定义本节可复用的核心函数。
        if not isinstance(prefix, list) or not prefix or any(type(token) is not int or not 0 <= token < self.target.vocab_size for token in prefix):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_prefix")  # 遇到非法合同立即显式失败。
        if prefix[-1] == self.eos_token:  # 按当前条件选择后续控制路径。
            raise ValueError("prefix_already_ended")  # 遇到非法合同立即显式失败。
        if not isinstance(max_new_tokens, int) or not 1 <= max_new_tokens <= 64:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_max_new_tokens")  # 遇到非法合同立即显式失败。
        if not isinstance(generator, torch.Generator):  # 按当前条件选择后续控制路径。
            raise TypeError("explicit_torch_generator_required")  # 遇到非法合同立即显式失败。
        self.target_cache.begin(request_id, prefix)  # 执行当前语句以推进本节示例。
        self.draft_cache.begin(request_id, prefix)  # 执行当前语句以推进本节示例。
        prefix_length = len(prefix)  # 计算并保存当前步骤的中间状态。
        trace = {"blocks": 0, "proposed": 0, "accepted": 0, "rejected": 0, "bonuses": 0, "rollbacks": 0}  # 计算并保存当前步骤的中间状态。

        while self.target_cache.position(request_id) - prefix_length < max_new_tokens:  # 在终止条件满足前持续推进状态。
            trace["blocks"] += 1  # 计算并保存当前步骤的中间状态。
            remaining = max_new_tokens - (self.target_cache.position(request_id) - prefix_length)  # 计算并保存当前步骤的中间状态。
            proposal_budget = min(self.gamma, remaining)  # 计算并保存当前步骤的中间状态。
            proposals, q_rows = [], []  # 计算并保存当前步骤的中间状态。
            for _ in range(proposal_budget):  # 遍历输入元素以累积或检查结果。
                current = self.draft_cache.tokens(request_id)[-1]  # 计算并保存当前步骤的中间状态。
                q = self.draft.next_probs(current)  # 计算并保存当前步骤的中间状态。
                proposal = sample_categorical57(q, generator)  # 计算并保存当前步骤的中间状态。
                proposals.append(proposal); q_rows.append(q)  # 执行当前语句以推进本节示例。
                self.draft_cache.append(request_id, proposal)  # 执行当前语句以推进本节示例。
                trace["proposed"] += 1  # 计算并保存当前步骤的中间状态。
                if proposal == self.eos_token:  # 按当前条件选择后续控制路径。
                    break  # 调整当前循环或占位控制流。

            rejected = False  # 计算并保存当前步骤的中间状态。
            for proposal, q in zip(proposals, q_rows):  # 遍历输入元素以累积或检查结果。
                current = self.target_cache.tokens(request_id)[-1]  # 计算并保存当前步骤的中间状态。
                p = self.target.next_probs(current)  # 计算并保存当前步骤的中间状态。
                uniform = float(torch.rand((), generator=generator))  # 计算并保存当前步骤的中间状态。
                if uniform < acceptance_probability57(p, q, proposal):  # 按当前条件选择后续控制路径。
                    self.target_cache.append(request_id, proposal)  # 执行当前语句以推进本节示例。
                    trace["accepted"] += 1  # 计算并保存当前步骤的中间状态。
                    if proposal == self.eos_token:  # 按当前条件选择后续控制路径。
                        break  # 调整当前循环或占位控制流。
                else:  # 处理前置条件不成立的分支。
                    replacement = sample_categorical57(residual_distribution57(p, q), generator)  # 计算并保存当前步骤的中间状态。
                    self.target_cache.append(request_id, replacement)  # 执行当前语句以推进本节示例。
                    keep_before_replacement = self.target_cache.position(request_id) - 1  # 计算并保存当前步骤的中间状态。
                    trace["rollbacks"] += self.draft_cache.rollback(request_id, keep_before_replacement)  # 计算并保存当前步骤的中间状态。
                    self.draft_cache.append(request_id, replacement)  # 执行当前语句以推进本节示例。
                    trace["rejected"] += 1  # 计算并保存当前步骤的中间状态。
                    rejected = True  # 计算并保存当前步骤的中间状态。
                    break  # 调整当前循环或占位控制流。

            generated = self.target_cache.tokens(request_id)[prefix_length:]  # 计算并保存当前步骤的中间状态。
            ended = bool(generated and generated[-1] == self.eos_token)  # 计算并保存当前步骤的中间状态。
            if not rejected and not ended and len(generated) < max_new_tokens:  # 按当前条件选择后续控制路径。
                p_bonus = self.target.next_probs(self.target_cache.tokens(request_id)[-1])  # 计算并保存当前步骤的中间状态。
                bonus = sample_categorical57(p_bonus, generator)  # 计算并保存当前步骤的中间状态。
                self.target_cache.append(request_id, bonus)  # 执行当前语句以推进本节示例。
                self.draft_cache.append(request_id, bonus)  # 执行当前语句以推进本节示例。
                trace["bonuses"] += 1  # 计算并保存当前步骤的中间状态。
                generated.append(bonus)  # 执行当前语句以推进本节示例。
                ended = bonus == self.eos_token  # 计算并保存当前步骤的中间状态。
            if ended:  # 按当前条件选择后续控制路径。
                break  # 调整当前循环或占位控制流。

        if self.target_cache.tokens(request_id) != self.draft_cache.tokens(request_id):  # 按当前条件选择后续控制路径。
            raise RuntimeError("target_draft_cache_diverged")  # 遇到非法合同立即显式失败。
        return self.target_cache.tokens(request_id)[prefix_length:prefix_length + max_new_tokens], trace  # 返回当前分支计算出的结果。

    def forward_batch(self, requests, max_new_tokens, generators):  # 定义本节可复用的核心函数。
        if len(requests) != len(generators):  # 按当前条件选择后续控制路径。
            raise ValueError("batch_generator_count_mismatch")  # 遇到非法合同立即显式失败。
        outputs = {}  # 计算并保存当前步骤的中间状态。
        for request, generator in zip(requests, generators):  # 遍历输入元素以累积或检查结果。
            request_id = request["request_id"]  # 计算并保存当前步骤的中间状态。
            outputs[request_id] = self(request["prefix"], max_new_tokens, generator, request_id)  # 计算并保存当前步骤的中间状态。
        return outputs  # 返回当前分支计算出的结果。

    def release_request(self, request_id):  # 定义本节可复用的核心函数。
        self.target_cache.release(request_id)  # 执行当前语句以推进本节示例。
        self.draft_cache.release(request_id)  # 执行当前语句以推进本节示例。

decoder57 = SpeculativeDecoder57(target_model57, draft_model57, gamma=2, eos_token=TOKENIZER57["<eos>"])  # 计算并保存当前步骤的中间状态。
generated57, trace57 = decoder57([0], 5, torch.Generator().manual_seed(71), "main")  # 计算并保存当前步骤的中间状态。
assert 1 <= len(generated57) <= 5  # 用受控断言验证关键不变量。
assert trace57["proposed"] >= trace57["accepted"]  # 用受控断言验证关键不变量。
assert trace57["rejected"] <= trace57["blocks"]  # 用受控断言验证关键不变量。
assert decoder57.target_cache.tokens("main") == decoder57.draft_cache.tokens("main")  # 用受控断言验证关键不变量。
assert decoder57.target_cache.position("main") == 1 + len(generated57)  # 用受控断言验证关键不变量。
decoder57.release_request("main")  # 执行当前语句以推进本节示例。

## 7. 全接受、全拒绝、bonus 与 EOS oracle

p=q 时每个 proposal 接受率为 1；若预算仍有空间，target bonus 必须出现。分布近乎不相交时，第一个 proposal 应拒绝，draft 超前 token 被 rollback，replacement 来自 p-q 正部。target/draft 都确定性指向 EOS 时，只生成一个 EOS 且没有 bonus。

In [ ]:
def deterministic_matrix57(next_token):  # 定义本节可复用的核心函数。
    matrix = torch.zeros(4, 4, dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
    matrix[:, next_token] = 1.0  # 计算并保存当前步骤的中间状态。
    return matrix  # 返回当前分支计算出的结果。

cycle57 = deterministic_matrix57(1)  # 计算并保存当前步骤的中间状态。
accept_decoder57 = SpeculativeDecoder57(BigramLM57(cycle57), BigramLM57(cycle57), gamma=2)  # 计算并保存当前步骤的中间状态。
accept_tokens57, accept_trace57 = accept_decoder57([0], 3, torch.Generator().manual_seed(10), "accept")  # 计算并保存当前步骤的中间状态。
assert accept_tokens57 == [1, 1, 1]  # 用受控断言验证关键不变量。
assert accept_trace57["accepted"] == 2  # 用受控断言验证关键不变量。
assert accept_trace57["rejected"] == 0 and accept_trace57["bonuses"] == 1  # 用受控断言验证关键不变量。
assert accept_trace57["rollbacks"] == 0  # 用受控断言验证关键不变量。

reject_decoder57 = SpeculativeDecoder57(BigramLM57(deterministic_matrix57(1)), BigramLM57(deterministic_matrix57(2)), gamma=3)  # 计算并保存当前步骤的中间状态。
reject_tokens57, reject_trace57 = reject_decoder57([0], 1, torch.Generator().manual_seed(11), "reject")  # 计算并保存当前步骤的中间状态。
assert reject_tokens57 == [1]  # 用受控断言验证关键不变量。
assert reject_trace57["accepted"] == 0 and reject_trace57["rejected"] == 1  # 用受控断言验证关键不变量。
assert reject_trace57["rollbacks"] >= 1  # 用受控断言验证关键不变量。
assert reject_decoder57.target_cache.tokens("reject") == reject_decoder57.draft_cache.tokens("reject")  # 用受控断言验证关键不变量。

eos57 = deterministic_matrix57(TOKENIZER57["<eos>"])  # 计算并保存当前步骤的中间状态。
eos_decoder57 = SpeculativeDecoder57(BigramLM57(eos57), BigramLM57(eos57), gamma=3)  # 计算并保存当前步骤的中间状态。
eos_tokens57, eos_trace57 = eos_decoder57([0], 6, torch.Generator().manual_seed(12), "eos")  # 计算并保存当前步骤的中间状态。
assert eos_tokens57 == [TOKENIZER57["<eos>"]]  # 用受控断言验证关键不变量。
assert eos_trace57["bonuses"] == 0  # 用受控断言验证关键不变量。
assert eos_trace57["accepted"] == 1  # 用受控断言验证关键不变量。

def add_mass57(table, key, mass):  # 定义本节可复用的核心函数。
    table[key] = table.get(key, 0.0) + float(mass)  # 计算并保存当前步骤的中间状态。

def enumerate_two_token_spec57(decoder, prefix_token):  # 定义本节可复用的核心函数。
    p0 = decoder.target.next_probs(prefix_token)  # 计算并保存当前步骤的中间状态。
    q0 = decoder.draft.next_probs(prefix_token)  # 计算并保存当前步骤的中间状态。
    residual0 = None if torch.allclose(p0, q0, atol=1e-15) else residual_distribution57(p0, q0)  # 计算并保存当前步骤的中间状态。
    outcomes = {}  # 计算并保存当前步骤的中间状态。
    for proposal in range(decoder.target.vocab_size):  # 遍历输入元素以累积或检查结果。
        accepted_mass = torch.minimum(p0[proposal], q0[proposal])  # 计算并保存当前步骤的中间状态。
        if accepted_mass > 0:  # 按当前条件选择后续控制路径。
            if proposal == decoder.eos_token:  # 按当前条件选择后续控制路径。
                add_mass57(outcomes, (proposal,), accepted_mass)  # 执行当前语句以推进本节示例。
            else:  # 处理前置条件不成立的分支。
                bonus_p = decoder.target.next_probs(proposal)  # 计算并保存当前步骤的中间状态。
                for bonus in range(decoder.target.vocab_size):  # 遍历输入元素以累积或检查结果。
                    add_mass57(outcomes, (proposal, bonus), accepted_mass * bonus_p[bonus])  # 执行当前语句以推进本节示例。
        rejected_mass = (q0[proposal] - p0[proposal]).clamp_min(0)  # 计算并保存当前步骤的中间状态。
        if rejected_mass > 0:  # 按当前条件选择后续控制路径。
            for replacement in range(decoder.target.vocab_size):  # 遍历输入元素以累积或检查结果。
                replacement_mass = rejected_mass * residual0[replacement]  # 计算并保存当前步骤的中间状态。
                if replacement_mass <= 0:  # 按当前条件选择后续控制路径。
                    continue  # 调整当前循环或占位控制流。
                if replacement == decoder.eos_token:  # 按当前条件选择后续控制路径。
                    add_mass57(outcomes, (replacement,), replacement_mass)  # 执行当前语句以推进本节示例。
                else:  # 处理前置条件不成立的分支。
                    p1 = decoder.target.next_probs(replacement)  # 计算并保存当前步骤的中间状态。
                    q1 = decoder.draft.next_probs(replacement)  # 计算并保存当前步骤的中间状态。
                    second = exact_first_output57(p1, q1)  # 计算并保存当前步骤的中间状态。
                    for token2 in range(decoder.target.vocab_size):  # 遍历输入元素以累积或检查结果。
                        add_mass57(outcomes, (replacement, token2), replacement_mass * second[token2])  # 执行当前语句以推进本节示例。
    return outcomes  # 返回当前分支计算出的结果。

def enumerate_two_token_target57(decoder, prefix_token):  # 定义本节可复用的核心函数。
    outcomes = {}  # 计算并保存当前步骤的中间状态。
    p0 = decoder.target.next_probs(prefix_token)  # 计算并保存当前步骤的中间状态。
    for first in range(decoder.target.vocab_size):  # 遍历输入元素以累积或检查结果。
        if first == decoder.eos_token:  # 按当前条件选择后续控制路径。
            add_mass57(outcomes, (first,), p0[first])  # 执行当前语句以推进本节示例。
        else:  # 处理前置条件不成立的分支。
            p1 = decoder.target.next_probs(first)  # 计算并保存当前步骤的中间状态。
            for second in range(decoder.target.vocab_size):  # 遍历输入元素以累积或检查结果。
                add_mass57(outcomes, (first, second), p0[first] * p1[second])  # 执行当前语句以推进本节示例。
    return outcomes  # 返回当前分支计算出的结果。

distribution_decoder57 = SpeculativeDecoder57(  # 计算并保存当前步骤的中间状态。
    BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=1, eos_token=TOKENIZER57["<eos>"]  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
exact_spec_two57 = enumerate_two_token_spec57(distribution_decoder57, TOKENIZER57["<bos>"])  # 计算并保存当前步骤的中间状态。
exact_target_two57 = enumerate_two_token_target57(distribution_decoder57, TOKENIZER57["<bos>"])  # 计算并保存当前步骤的中间状态。
all_outcomes57 = set(exact_spec_two57) | set(exact_target_two57)  # 计算并保存当前步骤的中间状态。
assert abs(sum(exact_spec_two57.values()) - 1.0) < 1e-7  # 用受控断言验证关键不变量。
assert abs(sum(exact_target_two57.values()) - 1.0) < 1e-7  # 用受控断言验证关键不变量。
assert max(abs(exact_spec_two57.get(key, 0.0) - exact_target_two57.get(key, 0.0)) for key in all_outcomes57) < 2e-7  # 用受控断言验证关键不变量。
assert any(len(key) == 1 and key[0] == TOKENIZER57["<eos>"] for key in exact_spec_two57)  # 用受控断言验证关键不变量。
assert any(len(key) == 2 for key in exact_spec_two57)  # 用受控断言验证关键不变量。

empirical_counts57 = {}  # 计算并保存当前步骤的中间状态。
for seed57 in range(500):  # 遍历输入元素以累积或检查结果。
    request_id57 = "dist-" + str(seed57)  # 计算并保存当前步骤的中间状态。
    tokens57, _ = distribution_decoder57(  # 计算并保存当前步骤的中间状态。
        [TOKENIZER57["<bos>"]], 2, torch.Generator().manual_seed(seed57), request_id57  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    distribution_decoder57.release_request(request_id57)  # 执行当前语句以推进本节示例。
    key57 = tuple(tokens57)  # 计算并保存当前步骤的中间状态。
    empirical_counts57[key57] = empirical_counts57.get(key57, 0) + 1  # 计算并保存当前步骤的中间状态。
empirical_max_error57 = max(  # 计算并保存当前步骤的中间状态。
    abs(empirical_counts57.get(key, 0) / 500 - exact_target_two57.get(key, 0.0))  # 执行当前语句以推进本节示例。
    for key in all_outcomes57  # 遍历输入元素以累积或检查结果。
)  # 执行当前语句以推进本节示例。
assert empirical_max_error57 < 0.07  # 用受控断言验证关键不变量。

## 8. Generator 可复现与 batch/request 位置隔离

随机源必须由调用方显式提供，不能混用 Python random、全局 torch RNG 与设备 RNG。相同模型、prefix 和 seed 应产生相同 token/trace。批请求只是教学版逐请求循环，但每个 request_id 有独立 cache 和独立 generator；不同 prefix 长度下位置仍等于 prefix_len+generated_len。

In [ ]:
deterministic_a57 = SpeculativeDecoder57(BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=2)  # 计算并保存当前步骤的中间状态。
deterministic_b57 = SpeculativeDecoder57(BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=2)  # 计算并保存当前步骤的中间状态。
output_a57 = deterministic_a57([0], 4, torch.Generator().manual_seed(99), "same")  # 计算并保存当前步骤的中间状态。
output_b57 = deterministic_b57([0], 4, torch.Generator().manual_seed(99), "same")  # 计算并保存当前步骤的中间状态。
assert output_a57 == output_b57  # 用受控断言验证关键不变量。

batch_decoder57 = SpeculativeDecoder57(BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=2)  # 计算并保存当前步骤的中间状态。
requests57 = [{"request_id": "batch-a", "prefix": [0]}, {"request_id": "batch-b", "prefix": [0, 1]}]  # 计算并保存当前步骤的中间状态。
batch_outputs57 = batch_decoder57.forward_batch(  # 计算并保存当前步骤的中间状态。
    requests57, 3, [torch.Generator().manual_seed(101), torch.Generator().manual_seed(202)]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert set(batch_outputs57) == {"batch-a", "batch-b"}  # 用受控断言验证关键不变量。
for request57 in requests57:  # 遍历输入元素以累积或检查结果。
    request_id57 = request57["request_id"]  # 计算并保存当前步骤的中间状态。
    generated_count57 = len(batch_outputs57[request_id57][0])  # 计算并保存当前步骤的中间状态。
    assert batch_decoder57.target_cache.position(request_id57) == len(request57["prefix"]) + generated_count57  # 用受控断言验证关键不变量。
    assert batch_decoder57.target_cache.tokens(request_id57) == batch_decoder57.draft_cache.tokens(request_id57)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    batch_decoder57([0], 1, torch.Generator().manual_seed(1), "batch-a")  # 执行当前语句以推进本节示例。
    raise AssertionError("live request id was reused")  # 遇到非法合同立即显式失败。
except RuntimeError as error57:  # 捕获预期异常并验证失败分支。
    assert str(error57) == "request_already_exists"  # 用受控断言验证关键不变量。
batch_decoder57.release_request("batch-a"); batch_decoder57.release_request("batch-b")  # 执行当前语句以推进本节示例。

## 9. 教学循环与生产并行验证的差异

本例逐 token 调 target，算法分布正确但没有性能优势。生产 speculative decoding 会让 draft 先生成 gamma 个 token，再用 target 一次并行 forward 验证整块；每层 KV cache 需要可提交/回滚的页表，连续批处理还要处理不同请求的接受长度。

收益取决于接受率、gamma、draft 延迟、target 并行效率和内存带宽。tokenizer、采样温度/top-k/top-p、logit processor、EOS 和随机数消费顺序必须完全对齐；只比较 greedy token 相等不足以证明 sampling 分布正确。

## 10. 可信发布：同时绑定 target、draft 与解码 recipe

state 摘要以 target:: 和 draft:: 命名空间绑定每个 key、dtype、shape、bytes。metadata 绑定完整 tokenizer、评估 prompts、split、gamma、EOS、最大输出、采样算法与 cache 语义。

包外 MappingProxy registry 是信任根。loader 返回 PublishedSpeculative57，包装器为每次请求创建显式 generator，并在 finally 中释放 cache，避免异常请求遗留状态污染下一次调用。

In [ ]:
def state_digest57(state):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        header = canonical_json57({"key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)}).encode()  # 计算并保存当前步骤的中间状态。
        raw = tensor.numpy().tobytes()  # 计算并保存当前步骤的中间状态。
        digest.update(len(header).to_bytes(8, "big")); digest.update(header)  # 执行当前语句以推进本节示例。
        digest.update(len(raw).to_bytes(8, "big")); digest.update(raw)  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

CONFIG57 = {"vocab_size": 4}  # 计算并保存当前步骤的中间状态。
RECIPE57 = {  # 计算并保存当前步骤的中间状态。
    "algorithm": "speculative_sampling_with_positive_residual",  # 执行当前语句以推进本节示例。
    "gamma": 2, "eos_token": 3, "max_new_tokens": 8,  # 执行当前语句以推进本节示例。
    "temperature": 1.0, "top_k": None, "top_p": None,  # 执行当前语句以推进本节示例。
    "explicit_generator": True, "rollback_unit": "committed_token_position",  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

def flatten_states57(target, draft):  # 定义本节可复用的核心函数。
    state = {}  # 计算并保存当前步骤的中间状态。
    for prefix, model in (("target", target), ("draft", draft)):  # 遍历输入元素以累积或检查结果。
        for key, value in model.state_dict().items():  # 遍历输入元素以累积或检查结果。
            state[prefix + "::" + key] = value.detach().cpu().clone()  # 计算并保存当前步骤的中间状态。
    return state  # 返回当前分支计算出的结果。

def release_digest57(package):  # 定义本节可复用的核心函数。
    envelope = {  # 计算并保存当前步骤的中间状态。
        "release_id": package["release_id"], "metadata": package["metadata"],  # 执行当前语句以推进本节示例。
        "state_digest": state_digest57(package["state"]),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return sha57(canonical_json57(envelope).encode())  # 返回当前分支计算出的结果。

def build_release57(target, draft):  # 定义本节可复用的核心函数。
    state = flatten_states57(target, draft)  # 计算并保存当前步骤的中间状态。
    package = {  # 计算并保存当前步骤的中间状态。
        "release_id": "spec-bigram-v1",  # 执行当前语句以推进本节示例。
        "metadata": {  # 执行当前语句以推进本节示例。
            "config": copy.deepcopy(CONFIG57), "tokenizer": copy.deepcopy(TOKENIZER57),  # 执行当前语句以推进本节示例。
            "data": copy.deepcopy(PROMPTS57), "split": copy.deepcopy(SPLIT57),  # 执行当前语句以推进本节示例。
            "recipe": copy.deepcopy(RECIPE57), "allowed_subject": "inference-lab",  # 执行当前语句以推进本节示例。
        },  # 执行当前语句以推进本节示例。
        "state": state, "state_digest": state_digest57(state),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    package["self_digest"] = release_digest57(package)  # 计算并保存当前步骤的中间状态。
    return package  # 返回当前分支计算出的结果。

RELEASE_PACKAGE57 = build_release57(target_model57, draft_model57)  # 计算并保存当前步骤的中间状态。
TRUSTED_RELEASES57 = MappingProxyType({"spec-bigram-v1": release_digest57(RELEASE_PACKAGE57)})  # 计算并保存当前步骤的中间状态。

class PublishedSpeculative57(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, target, draft, metadata, subject):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.decoder = SpeculativeDecoder57(target, draft, metadata["recipe"]["gamma"], metadata["recipe"]["eos_token"])  # 计算并保存当前步骤的中间状态。
        self.tokenizer = MappingProxyType(copy.deepcopy(metadata["tokenizer"]))  # 计算并保存当前步骤的中间状态。
        self.subject = subject  # 计算并保存当前步骤的中间状态。
        self.max_new_tokens = metadata["recipe"]["max_new_tokens"]  # 计算并保存当前步骤的中间状态。

    def forward(self, prefix, max_new_tokens, seed, request_id):  # 定义本节可复用的核心函数。
        if self.subject != "inference-lab":  # 按当前条件选择后续控制路径。
            raise PermissionError("subject_not_authorized")  # 遇到非法合同立即显式失败。
        if not isinstance(seed, int):  # 按当前条件选择后续控制路径。
            raise TypeError("seed_must_be_int")  # 遇到非法合同立即显式失败。
        if max_new_tokens > self.max_new_tokens:  # 按当前条件选择后续控制路径。
            raise ValueError("request_exceeds_published_limit")  # 遇到非法合同立即显式失败。
        generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
        try:  # 尝试执行可能失败的受控操作。
            return self.decoder(prefix, max_new_tokens, generator, request_id)  # 返回当前分支计算出的结果。
        finally:  # 无论结果如何都执行收尾逻辑。
            if request_id in self.decoder.target_cache._requests:  # 按当前条件选择后续控制路径。
                self.decoder.release_request(request_id)  # 执行当前语句以推进本节示例。

def validate_metadata57(metadata):  # 定义本节可复用的核心函数。
    tokenizer = metadata["tokenizer"]  # 计算并保存当前步骤的中间状态。
    if tokenizer != TOKENIZER57 or sorted(tokenizer.values()) != list(range(len(tokenizer))):  # 按当前条件选择后续控制路径。
        raise ValueError("tokenizer_contract_mismatch")  # 遇到非法合同立即显式失败。
    data, split = metadata["data"], metadata["split"]  # 计算并保存当前步骤的中间状态。
    ids, referenced = [row["id"] for row in data], sum(split.values(), [])  # 计算并保存当前步骤的中间状态。
    if len(ids) != len(set(ids)) or sorted(ids) != sorted(referenced) or len(referenced) != len(set(referenced)):  # 按当前条件选择后续控制路径。
        raise ValueError("data_split_contract_mismatch")  # 遇到非法合同立即显式失败。
    if any(not row["tokens"] or any(type(token) is not int or token not in tokenizer.values() for token in row["tokens"]) for row in data):  # 按当前条件选择后续控制路径。
        raise ValueError("invalid_bound_prompts")  # 遇到非法合同立即显式失败。
    if metadata["recipe"] != RECIPE57 or metadata["config"]["vocab_size"] != len(tokenizer):  # 按当前条件选择后续控制路径。
        raise ValueError("decoder_recipe_mismatch")  # 遇到非法合同立即显式失败。

def load_published57(package, subject):  # 定义本节可复用的核心函数。
    release_id = package.get("release_id")  # 计算并保存当前步骤的中间状态。
    actual = release_digest57(package)  # 计算并保存当前步骤的中间状态。
    if release_id not in TRUSTED_RELEASES57 or actual != TRUSTED_RELEASES57[release_id]:  # 按当前条件选择后续控制路径。
        raise PermissionError("untrusted_release_digest")  # 遇到非法合同立即显式失败。
    if package.get("self_digest") != actual or package.get("state_digest") != state_digest57(package["state"]):  # 按当前条件选择后续控制路径。
        raise ValueError("corrupt_release")  # 遇到非法合同立即显式失败。
    validate_metadata57(package["metadata"])  # 执行当前语句以推进本节示例。
    if subject != package["metadata"]["allowed_subject"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("subject_not_authorized")  # 遇到非法合同立即显式失败。
    vocab_size = package["metadata"]["config"]["vocab_size"]  # 计算并保存当前步骤的中间状态。
    uniform = torch.full((vocab_size, vocab_size), 1.0 / vocab_size, dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
    target, draft = BigramLM57(uniform), BigramLM57(uniform)  # 计算并保存当前步骤的中间状态。
    target_state = {key.split("::", 1)[1]: value for key, value in package["state"].items() if key.startswith("target::")}  # 计算并保存当前步骤的中间状态。
    draft_state = {key.split("::", 1)[1]: value for key, value in package["state"].items() if key.startswith("draft::")}  # 计算并保存当前步骤的中间状态。
    target.load_state_dict(target_state, strict=True); draft.load_state_dict(draft_state, strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedSpeculative57(target, draft, package["metadata"], subject)  # 返回当前分支计算出的结果。

published57 = load_published57(copy.deepcopy(RELEASE_PACKAGE57), "inference-lab")  # 计算并保存当前步骤的中间状态。
published_tokens57, published_trace57 = published57([0], 4, 303, "published-a")  # 计算并保存当前步骤的中间状态。
assert 1 <= len(published_tokens57) <= 4  # 用受控断言验证关键不变量。
assert published_trace57["blocks"] >= 1  # 用受控断言验证关键不变量。
assert published57.decoder.target_cache._requests == {}  # 用受控断言验证关键不变量。
assert published57.decoder.draft_cache._requests == {}  # 用受控断言验证关键不变量。
assert isinstance(TRUSTED_RELEASES57, MappingProxyType)  # 用受控断言验证关键不变量。

## 11. 整体重签、非法请求与失败恢复

下列攻击同时修改 target state、更新 state_digest 和 self_digest，包内字段完全一致；包外 registry 仍拒绝。包装器还拒绝超出发布上限的输出，并确认即使 forward 抛错，finally 也不会遗留请求 cache。

In [ ]:
forged57 = copy.deepcopy(RELEASE_PACKAGE57)  # 计算并保存当前步骤的中间状态。
forged_key57 = next(key for key in forged57["state"] if key.startswith("target::"))  # 计算并保存当前步骤的中间状态。
forged57["state"][forged_key57].view(-1)[0] += 0.25  # 计算并保存当前步骤的中间状态。
forged57["state_digest"] = state_digest57(forged57["state"])  # 计算并保存当前步骤的中间状态。
forged57["self_digest"] = release_digest57(forged57)  # 计算并保存当前步骤的中间状态。
assert forged57["self_digest"] == release_digest57(forged57)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    load_published57(forged57, "inference-lab")  # 执行当前语句以推进本节示例。
    raise AssertionError("fully resigned forged decoder was trusted")  # 遇到非法合同立即显式失败。
except PermissionError as error57:  # 捕获预期异常并验证失败分支。
    assert str(error57) == "untrusted_release_digest"  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    load_published57(RELEASE_PACKAGE57, "outsider")  # 执行当前语句以推进本节示例。
    raise AssertionError("unauthorized subject was accepted")  # 遇到非法合同立即显式失败。
except PermissionError as error57:  # 捕获预期异常并验证失败分支。
    assert str(error57) == "subject_not_authorized"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    published57([0], 9, 1, "too-long")  # 执行当前语句以推进本节示例。
    raise AssertionError("request above published limit was accepted")  # 遇到非法合同立即显式失败。
except ValueError as error57:  # 捕获预期异常并验证失败分支。
    assert str(error57) == "request_exceeds_published_limit"  # 用受控断言验证关键不变量。
assert published57.decoder.target_cache._requests == {}  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    published57([3], 2, 1, "ended")  # 执行当前语句以推进本节示例。
    raise AssertionError("already-ended prefix was accepted")  # 遇到非法合同立即显式失败。
except ValueError as error57:  # 捕获预期异常并验证失败分支。
    assert str(error57) == "prefix_already_ended"  # 用受控断言验证关键不变量。
assert published57.decoder.draft_cache._requests == {}  # 用受控断言验证关键不变量。

## 12. 结论、复杂度与原始资料

本例先精确穷举每个上下文的首输出，再通过实际 decoder 的两 token 路径解析枚举与 500 次采样验证 EOS、拒绝续块和 bonus 的联合分布等于 target，并覆盖 p/q、q=0、全接受、全拒绝、bonus、EOS、显式 generator、批请求位置和 rollback。算法减少的是昂贵 target 的串行调用次数，不改变 target 分布；如果 residual、随机数或 cache 提交任一处写错，就可能只在采样模式下悄悄偏分布。

原始资料：

- Fast Inference from Transformers via Speculative Decoding：https://arxiv.org/abs/2211.17192
- Accelerating Large Language Model Decoding with Speculative Sampling：https://arxiv.org/abs/2302.01318
- PyTorch multinomial 官方文档：https://pytorch.org/docs/stable/generated/torch.multinomial.html